In [1]:
import numpy as np

In [ ]:
def softmax(Z):
    Z = Z - np.max(Z, axis=-1, keepdims=True)
    exp = np.exp(Z)
    return exp / np.sum(exp, axis=-1, keepdims=True)


def multihead_attention(X, W_Q, W_K, W_V, W_O, h, mask=None):

    N, seq_len, d_model = X.shape
    d_k = d_model // h

    # Projection
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V

    # Multi-Head Transformation
    multihead_Q = Q.reshape(N, seq_len, h, d_k)
    multihead_K = K.reshape(N, seq_len, h, d_k)
    multihead_V = V.reshape(N, seq_len, h, d_k)

    multihead_Q = multihead_Q.transpose(0, 2, 1, 3)
    multihead_K = multihead_K.transpose(0, 2, 1, 3)
    multihead_V = multihead_V.transpose(0, 2, 1, 3)

    # Calculating Q @ K
    scores = multihead_Q @ multihead_K.transpose(0, 1, 3, 2)
    scaled_scores = scores / np.sqrt(d_k)

    if mask is not None:
        scaled_scores += mask

    weights = softmax(scaled_scores)
    mha_output = weights @ V
    concat_output = mha_output.transpose(0, 2, 1, 3).reshape(N, seq_len, d_model)
    output = concat_output @ W_O
    return output


def layer_norm(X, gamma, beta, eps=1e-5):

    mean = X.mean(axis=-1, keepdims=True)
    std = X.std(axis=-1, keepdims=True)

    norm = ((X - mean)/std + eps)*gamma + beta
    return norm

def add_and_norm(X, sublayer_output, gamma, beta):

    return layer_norm(X + sublayer_output, gamma, beta)

def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))


def FFN(X, W1, b1, W2, b2):
    
    Z1 = X @ W1 + b1
    a1 = gelu(Z1)
    Z2 = a1 @ W2 + b2
    return Z2


def decoder(X, W_Q, W_K, W_V, W_O, h, W1, b1, W2, b2, gamma1, beta1, gamma2, beta2, mask):

    mha_output = multihead_attention(X, W_Q, W_K, W_V, W_O, h, mask)
    sub_layer1 = add_and_norm(X, mha_output, gamma1, beta1)

    ffn_output = FFN(sub_layer1, W1, b1, W2, b2)
    sub_layer2 = add_and_norm(sub_layer1, ffn_output, gamma2, beta2)

    return sub_layer2